# Dos features para mi proyecto

Proyecto: **GrantMatch**. Archivo: `../data/processed/obt_award_work_muestra.csv`, el mismo de [`explorar_proyecto.ipynb`](./explorar_proyecto.ipynb).

Aquí pruebo las features antes de decidir cuáles se quedan. Las que se quedan pasan a [`pipeline_proyecto.py`](./pipeline_proyecto.py); lo que queda en este notebook es la evidencia de por qué las elegí. Antes de hacer commit: **Restart** y **Run All**.

## Paso 1: el momento de la predicción

GrantMatch predice si una convocatoria (award) es pertinente para un investigador, representado por una de sus publicaciones: la probabilidad de que ese award financie ese trabajo. Se predice en el momento de recomendar, cuando la convocatoria está abierta.

- **Existe en ese instante:** los datos del award (funder, país del funder, tipo de financiamiento, campo, monto, duración, año de inicio) y los del perfil del investigador (campo de sus publicaciones, países de sus autores, número de autores, citas).
- **Todavía no existe:** si el award lo financió (`label`, lo que predecimos), cuántas publicaciones terminó financiando (`funded_outputs_count`) y las publicaciones que salgan del financiamiento (por eso `publication_year` tampoco entra: en un par real la publicación es posterior al award).

In [ ]:
import numpy as np
import pandas as pd

datos = pd.read_csv("../data/processed/obt_award_work_muestra.csv")
datos.columns = datos.columns.str.lower()
OBJETIVO = "label"
print("Muestra de la OBT:", datos.shape, " valores de label:", datos[OBJETIVO].value_counts().to_dict())

### Antes de las features: pares negativos

En la OBT todas las filas son financiamientos reales (`label` = 1). Para tener algo que predecir, a cada award le asigno al azar una publicación de otra fila que **no** financió (`label` = 0). Un par negativo conserva las columnas del award y toma las de la publicación de otra fila. Los pares quedan en `../data/processed/pares_proyecto.csv`, que lee `pipeline_proyecto.py`.

In [ ]:
COLUMNAS_AWARD = [
    "award_key", "funder_name", "funder_country_code", "is_global_south", "funding_type",
    "award_field_name", "award_domain_name", "award_start_year", "amount_usd",
    "is_amount_invalid", "duration_days", "funded_outputs_count", "has_description", "award_text_len",
]
COLUMNAS_PUBLICACION = [
    "work_key", "work_field_name", "work_domain_name", "publication_year", "has_abstract",
    "work_text_len", "cited_by_count", "fwci", "authors_count", "linked_authors_count",
    "language", "work_type", "author_country_codes",
]

positivos = datos[COLUMNAS_AWARD + COLUMNAS_PUBLICACION].copy()
positivos[OBJETIVO] = 1
pares_reales = set(zip(positivos["award_key"], positivos["work_key"]))

generador = np.random.default_rng(42)
indices = generador.permutation(len(positivos))
for _ in range(10):  # vuelve a sortear los que cayeron en un par real
    choque = np.array([(a, w) in pares_reales for a, w in
                       zip(positivos["award_key"].values, positivos["work_key"].values[indices])])
    if not choque.any():
        break
    indices[choque] = generador.permutation(indices[choque])

negativos = pd.concat([
    positivos[COLUMNAS_AWARD].reset_index(drop=True),
    positivos[COLUMNAS_PUBLICACION].iloc[indices].reset_index(drop=True),
], axis=1)
negativos[OBJETIVO] = 0
negativos = negativos[[(a, w) not in pares_reales for a, w in zip(negativos["award_key"], negativos["work_key"])]]

pares = pd.concat([positivos, negativos], ignore_index=True)
pares.to_csv("../data/processed/pares_proyecto.csv", index=False)
print("Pares:", pares.shape, pares[OBJETIVO].value_counts().to_dict())

## Paso 2: feature 1

In [ ]:
pares["mismo_campo"] = (
    pares["award_field_name"].notna()
    & (pares["award_field_name"] == pares["work_field_name"])
).astype(int)

print("Correlacion con label:", round(pares[["mismo_campo", OBJETIVO]].corr().iloc[0, 1], 3))
pares.groupby("mismo_campo")[OBJETIVO].agg(["mean", "size"]).round(3)

**`mismo_campo`** (1 si el award y la publicación son del mismo campo de investigación). Sale del Paso 4 de la exploración: ⟨X⟩ % de los pares reales comparten campo. **Evidencia:** entre los pares con `mismo_campo` = 1, ⟨X⟩ % son reales, contra ⟨X⟩ % con `mismo_campo` = 0 (correlación con `label` de ⟨X⟩). **Existe al predecir:** el campo del award viene de la convocatoria y el de la publicación, del perfil del investigador. Es binaria, así que entra tal cual en cualquier modelo.

Ojo: por cómo se arman los negativos, las columnas de un solo lado (solo del award o solo de la publicación) tienen la misma distribución en pares reales y negativos. La señal está en las columnas que comparan los dos lados, como esta.

## Paso 3: feature 2

In [ ]:
paises_autores = pares["author_country_codes"].fillna("").astype(str).str.upper().str.split("|")
pais_funder = pares["funder_country_code"].fillna("").astype(str).str.upper()
pares["funder_en_pais_autor"] = [
    int(bool(pais) and pais in paises) for pais, paises in zip(pais_funder, paises_autores)
]

print("Correlacion con label:", round(pares[["funder_en_pais_autor", OBJETIVO]].corr().iloc[0, 1], 3))
pares.groupby("funder_en_pais_autor")[OBJETIVO].agg(["mean", "size"]).round(3)

**`funder_en_pais_autor`** (1 si el país del funder está entre los países de los autores). Sale del conocimiento del dominio: la mayoría de las agencias financian sobre todo a investigadores de su propio país. **Evidencia:** entre los pares con `funder_en_pais_autor` = 1, ⟨X⟩ % son reales, contra ⟨X⟩ % con 0 (correlación con `label` de ⟨X⟩). **Existe al predecir:** el país del funder es parte de la convocatoria y el de los autores, de su afiliación. También es binaria. Si en vez de esto usara el país del funder como categoría, la codificaría con one-hot para un modelo lineal, y ordinal solo para árboles.

In [ ]:
pares.head()